# API 调用与检索

用设备说明试关键词检索和语义检索，再查看基于资料的回答。函数整理在 `main.py`。语义检索部分可以单独运行，不需要前面的 DeepSeek 客户端。

In [ ]:
from main import records, create_client, ask_model, find_records, build_context, answer_question

## 直接提问

输入密钥后修改问题试一下。运行下一格会请求 DeepSeek。

In [ ]:
client = create_client()
question = "Python列表和字典有什么区别？"
print(ask_model(question, None, client))

## 先看检索结果

检索返回记录列表，整理后得到保留来源的字符串。这格不请求模型。

In [ ]:
hits = find_records(records, "风机")
context = build_context(hits)
print(context)

## 根据找到的资料回答

分别检查：有答案、资料没写答案、没有相关资料。前两组会请求模型，最后一组直接返回。

In [ ]:
cases = [
    ("多久检查风机滤网？", "滤网"),
    ("风机的生产厂家是谁？", "风机"),
    ("电机的额定功率是多少？", "电机"),
]
for question, keyword in cases:
    print("问题：", question)
    print("回答：", answer_question(records, question, keyword, client))
    print()

## 换个关键词

‘过滤网’在原文中没有完整出现。这种查找无法识别它和‘滤网’的关系。

In [ ]:
print(find_records(records, "过滤网"))
assert find_records(records, "") == []
assert build_context([]) == ""

## 语义检索

用 BGE 中文模型把四段原文编码成向量。首次运行会下载模型，使用 CPU；后面几格只检索，不请求 DeepSeek。

In [ ]:
from main import records, load_embedder, semantic_search, get_records

model = load_embedder()
texts = [record["text"] for record in records]
doc_vectors = model.encode(texts, normalize_embeddings=True)
print(doc_vectors.shape)

### 用‘过滤网’找‘滤网’

问题和原文编码后比较余弦相似度。`corpus_id` 用来取回原始记录，`score` 用来排序。

In [ ]:
question = "过滤网多久维护一次？"
hits = semantic_search(records, question, model, doc_vectors, top_k=2)
for hit, record in zip(hits, get_records(records, hits)):
    print(f"{hit['score']:.4f} | {record['source']} | {record['text']}")

### 换三个问题

观察段落里是否包含答案。资料库非空时，取前两条通常仍会得到结果；相似度高不代表原文一定能回答问题。

In [ ]:
questions = [
    "机器太热应该怎么处理？",
    "水泵密封多久检查一次？",
    "设备的生产厂家是谁？",
]
for question in questions:
    hits = semantic_search(records, question, model, doc_vectors)
    print("问题：", question)
    for record in get_records(records, hits):
        print(record["source"], record["text"])
    print()

assert get_records(records, []) == []

## 接通语义检索与模型回答

把匹配记录整理成带来源的文字，再交给 DeepSeek。匹配记录是包含字典的列表；参考文字和回答都是字符串。

In [ ]:
from main import records, load_embedder, semantic_find, build_context, create_client, answer_with_search

if "model" not in globals():
    model = load_embedder()
question = "过滤网多久维护一次？"
matched_records = semantic_find(records, question, model)
context = build_context(matched_records)
print(context)

### 看原文，再看回答

下一格会请求 DeepSeek 一次。对照上面的原文，检查回答和来源；有检索结果也可能没有答案。

In [ ]:
if "client" not in globals():
    client = create_client()
print(answer_with_search(records, question, model, client))

### 换三个问题

下一格会请求 DeepSeek 三次。过热问题在保存的练习中有一次返回“原文未提供”，所以这里一起打印候选原文，方便核对。

In [ ]:
questions = [
    "机器太热应该怎么处理？",
    "水泵密封多久检查一次？",
    "设备的生产厂家是谁？",
]
for question in questions:
    matched_records = semantic_find(records, question, model)
    print("问题：", question)
    print("参考资料：\n", build_context(matched_records))
    print("回答：", answer_with_search(records, question, model, client))
    print()

### 提前结束的情况

空资料和空问题直接返回提示，这格不请求模型。

In [ ]:
print(answer_with_search([], "多久检查滤网？", model, client))
print(answer_with_search(records, "", model, client))

## 从 TXT 读取资料

两份模拟说明，每个非空行是一条记录。来源保留文件路径和原始行号。已有文件会保留，方便修改后重新读取。

In [ ]:
from pathlib import Path
from main import load_records, load_embedder, semantic_find, build_context, create_client, answer_with_search

document_dir = Path("practice_documents05")
document_dir.mkdir(exist_ok=True)
file_contents = {
    "风机说明.txt": "风机A每30天检查一次滤网。\n\n风机A过热时，应先停机，再检查风道。\n风机A启动前应确认防护罩安装牢固。\n",
    "水泵说明.txt": "水泵B每90天检查一次密封。\n水泵B出现异常噪声时，应先停机检查。\n\n水泵B首次投入运行前，应确认管路连接完好。\n",
}
for name, content in file_contents.items():
    path = document_dir / name
    if not path.exists():
        path.write_text(content, encoding="utf-8")
filenames = [document_dir / name for name in file_contents]
file_records = load_records(filenames)
print("记录数量：", len(file_records))
for record in file_records:
    print(record)
print("空文件列表：", load_records([]))

### 查看文件里的候选资料

这格只检索，首次运行可能下载模型，不请求 DeepSeek。

In [ ]:
if "model" not in globals():
    model = load_embedder()
question = "风机A滤网多久检查一次？"
print(build_context(semantic_find(file_records, question, model)))

### 用文件中的原文回答

下一格请求 DeepSeek 三次。对照来源文件核对回答。

In [ ]:
if "client" not in globals():
    client = create_client()
for question in ["风机A滤网多久检查一次？", "风机A过热时应该怎么处理？", "风机A的生产厂家是谁？"]:
    print("问题：", question)
    print("回答：", answer_with_search(file_records, question, model, client))
    print()

### 修改文件后重新读取

在 VS Code 打开下面打印的文件，把第一行的30改为45并保存，然后运行最后一格。保存的原练习中，文件和回答仍是30天，这一步还需验证。

In [ ]:
print(filenames[0].resolve())

下面会请求 DeepSeek 一次。先看重新读取的第一条原文是否已变成45天，再核对回答；旧列表不会随着文件修改自动更新。

In [ ]:
file_records = load_records(filenames)
print("重新读取的第一条原文：", file_records[0]["text"])
print(answer_with_search(file_records, "风机A滤网多久检查一次？", model, client))